In [1]:
# Cell 1: Mount Google Drive and unpack staged 4-stem datasets
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

# Create local scratch directories on Colab NVMe
os.makedirs('/content/data/staged_train_4stem', exist_ok=True)
os.makedirs('/content/data/staged_val_concert03', exist_ok=True)

# Unpack training set
print("Unzipping 4-stem training set to local Colab NVMe...")
!unzip -q /content/drive/MyDrive/sanidha_train_4stem.zip -d /content/data/staged_train_4stem/

# Unpack validation set (Concert 03)
print("Unzipping Concert 03 validation set...")
!unzip -q /content/drive/MyDrive/sanidha_val_concert03.zip -d /content/data/staged_val_concert03/

print("Data unpacking complete!")

Mounted at /content/drive
Unzipping 4-stem training set to local Colab NVMe...
Unzipping Concert 03 validation set...
Data unpacking complete!


In [2]:
# Cell 2: Verify manifest and dataset statistics
import pandas as pd
from pathlib import Path

train_manifest = pd.read_csv('/content/data/staged_train_4stem/manifest.csv')
val_manifest = pd.read_csv('/content/data/staged_val_concert03/manifest.csv')

print(f"Train chunks: {len(train_manifest)} | Concerts: {sorted(train_manifest['concert_id'].unique())}")
print(f"Val chunks:   {len(val_manifest)} | Concerts: {sorted(val_manifest['concert_id'].unique())}")
print("\nTrain has_ghatam breakdown:\n", train_manifest['has_ghatam'].value_counts())
print("\nVal has_ghatam breakdown:\n", val_manifest['has_ghatam'].value_counts())

# Strict zero-leakage assertion
assert 3 not in train_manifest['concert_id'].unique(), "LEAKAGE: Concert 3 detected in train set!"
assert set(val_manifest['concert_id'].unique()) == {3}, "VAL SPLIT ERROR: Concert 3 missing or contaminated!"
print("\nSplit validation: PASSED (Zero leakage confirmed).")

Train chunks: 2920 | Concerts: [np.int64(1), np.int64(4), np.int64(5)]
Val chunks:   820 | Concerts: [np.int64(3)]

Train has_ghatam breakdown:
 has_ghatam
0    2360
1     560
Name: count, dtype: int64

Val has_ghatam breakdown:
 has_ghatam
1    820
Name: count, dtype: int64

Split validation: PASSED (Zero leakage confirmed).


In [3]:
# Cell 3: Path resolution check & automatic backslash normalization if needed
from pathlib import Path
import os
import pandas as pd

train_dir = Path("/content/data/staged_train_4stem")
manifest = pd.read_csv(train_dir / "manifest.csv")
sample_mix_rel = manifest.iloc[0]["mix_path"]
sample_mix_path = train_dir / sample_mix_rel

if sample_mix_path.exists():
    print(f"Path check PASSED: '{sample_mix_rel}' exists on Linux disk.")
else:
    print(f"Backslash path detected. Normalizing directory layout...")
    # Fix backslash-named files created by Windows zip on Linux
    import glob
    for p in train_dir.glob("**/*"):
        if "\\" in p.name:
            new_rel = p.name.replace("\\", "/")
            dest = train_dir / new_rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            p.rename(dest)
    # Check again
    assert (train_dir / sample_mix_rel).exists(), f"Still cannot find {sample_mix_rel}"
    print("Normalization complete! All files resolved to POSIX directories.")

Path check PASSED: 'chunks/C01_01-Vanajakshi-Varnam_chunk0000/mix.wav' exists on Linux disk.


In [4]:
# Cell 4: 4-Stem Sanidha PyTorch Dataset with Ghatam Loss Masking
import torch
from torch.utils.data import Dataset, DataLoader
import soundfile as sf
import numpy as np

class Sanidha4StemDataset(Dataset):
    def __init__(self, manifest_path: str | Path, root_dir: str | Path | None = None):
        self.manifest = pd.read_csv(manifest_path)
        self.root_dir = Path(root_dir) if root_dir else Path(manifest_path).parent

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        row = self.manifest.iloc[idx]

        # Load mix (2, N)
        mix, _ = sf.read(self.root_dir / row["mix_path"], dtype="float32", always_2d=True)
        mix_tensor = torch.from_numpy(mix.T)  # (2, T)

        # Load 4 stems: [vocals, violin, mridangam, ghatam]
        stems = []
        for col in ["vocals_path", "violin_path", "mridangam_path", "ghatam_path"]:
            audio, _ = sf.read(self.root_dir / row[col], dtype="float32", always_2d=True)
            stems.append(audio.T)

        targets_tensor = torch.from_numpy(np.stack(stems, axis=0))  # (4, 2, T)

        # Mask: 1 for active stems, 0 for absent stems (ghatam on C04/C05/C01-S07)
        # Stems: [vocals=1, violin=1, mridangam=1, ghatam=has_ghatam]
        mask_tensor = torch.tensor([1.0, 1.0, 1.0, float(row["has_ghatam"])], dtype=torch.float32)

        return mix_tensor, targets_tensor, mask_tensor

train_ds = Sanidha4StemDataset("/content/data/staged_train_4stem/manifest.csv")
val_ds = Sanidha4StemDataset("/content/data/staged_val_concert03/manifest.csv")

mix_sample, targets_sample, mask_sample = train_ds[0]
print(f"Mix shape:     {mix_sample.shape} (2, 264600)")
print(f"Targets shape: {targets_sample.shape} (4, 2, 264600)")
print(f"Mask sample:   {mask_sample.tolist()} ([voc, vln, mri, gha])")

Mix shape:     torch.Size([2, 264600]) (2, 264600)
Targets shape: torch.Size([4, 2, 264600]) (4, 2, 264600)
Mask sample:   [1.0, 1.0, 1.0, 1.0] ([voc, vln, mri, gha])


In [5]:
# Cell 5: HDemucs4Stem Model and Masked Hybrid Loss
!pip install -q demucs-infer julius

import torch.nn as nn
import demucs_infer.pretrained

class HDemucs4Stem(nn.Module):
    def __init__(self, freeze_encoders: bool = True):
        super().__init__()
        bag = demucs_infer.pretrained.get_model("htdemucs")
        self.backbone = bag.models[0] if hasattr(bag, "models") else bag

        # Western stems: 0=drums, 1=bass, 2=other, 3=vocals
        # Carnatic stems: 0=vocals, 1=violin, 2=mridangam, 3=ghatam
        # Direct channel reassignment: decoders learn instrument mapping
        if freeze_encoders:
            for p in self.backbone.encoder.parameters():
                p.requires_grad = False
            for p in self.backbone.tencoder.parameters():
                p.requires_grad = False

    def forward(self, x):
        # x: (B, 2, T) -> out: (B, 4, 2, T)
        return self.backbone(x)

class MaskedHybridLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.L1Loss(reduction='none')

    def forward(self, pred, target, mask):
        # pred: (B, 4, 2, T), target: (B, 4, 2, T), mask: (B, 4)
        l1_err = self.l1(pred, target).mean(dim=[-1, -2])  # (B, 4)
        masked_loss = (l1_err * mask).sum() / (mask.sum() + 1e-8)
        return masked_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HDemucs4Stem(freeze_encoders=True).to(device)
criterion = MaskedHybridLoss()
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-4)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Device: {device}")
print(f"Trainable params (decoders): {trainable_params:,}")
print(f"Frozen params (encoders):   {frozen_params:,}")

Device: cuda
Trainable params (decoders): 39,135,344
Frozen params (encoders):   2,849,112


In [6]:
# Cell 6: 4-Stem Fine-Tuning Loop with Masked Loss
import time
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR

batch_size = 4
num_epochs = 5

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

scaler = GradScaler()
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs * len(train_loader), eta_min=1e-5)

print(f"Starting 4-stem fine-tuning for {num_epochs} epochs...")
print(f"Total training batches per epoch: {len(train_loader)}")

best_val_loss = float("inf")
checkpoint_path = "/content/drive/MyDrive/hdemucs_sanidha_4stem_epoch5.pt"

for epoch in range(1, num_epochs + 1):
    model.train()
    running_train_loss = 0.0
    t0 = time.time()

    for batch_idx, (mix, targets, mask) in enumerate(train_loader):
        mix = mix.to(device)         # (B, 2, T)
        targets = targets.to(device) # (B, 4, 2, T)
        mask = mask.to(device)       # (B, 4)

        optimizer.zero_grad()
        with autocast():
            preds = model(mix)
            loss = criterion(preds, targets, mask)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_train_loss += loss.item()
        if (batch_idx + 1) % 150 == 0 or (batch_idx + 1) == len(train_loader):
            print(f"Epoch [{epoch}/{num_epochs}] | Batch [{batch_idx+1}/{len(train_loader)}] | Train Loss: {loss.item():.4f}")

    epoch_train_loss = running_train_loss / len(train_loader)

    # Validation loop
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for mix, targets, mask in val_loader:
            mix = mix.to(device)
            targets = targets.to(device)
            mask = mask.to(device)
            with autocast():
                preds = model(mix)
                val_loss = criterion(preds, targets, mask)
            running_val_loss += val_loss.item()

    epoch_val_loss = running_val_loss / len(val_loader)
    elapsed = time.time() - t0
    print(f"--- Epoch {epoch} Complete | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Time: {elapsed/60:.2f} mins ---")

    # Save checkpoint
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": epoch_train_loss,
        "val_loss": epoch_val_loss,
    }, checkpoint_path)
    print(f"Saved checkpoint to: {checkpoint_path}")

print("\n4-Stem Training Complete!")

Starting 4-stem fine-tuning for 5 epochs...
Total training batches per epoch: 730


/tmp/ipykernel_4643/3768504942.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_4643/3768504942.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/5] | Batch [150/730] | Train Loss: 0.0048
Epoch [1/5] | Batch [300/730] | Train Loss: 0.0025
Epoch [1/5] | Batch [450/730] | Train Loss: 0.0039
Epoch [1/5] | Batch [600/730] | Train Loss: 0.0063
Epoch [1/5] | Batch [730/730] | Train Loss: 0.0014


/tmp/ipykernel_4643/3768504942.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


--- Epoch 1 Complete | Train Loss: 0.0046 | Val Loss: 0.0027 | Time: 11.30 mins ---
Saved checkpoint to: /content/drive/MyDrive/hdemucs_sanidha_4stem_epoch5.pt
Epoch [2/5] | Batch [150/730] | Train Loss: 0.0027
Epoch [2/5] | Batch [300/730] | Train Loss: 0.0044
Epoch [2/5] | Batch [450/730] | Train Loss: 0.0028
Epoch [2/5] | Batch [600/730] | Train Loss: 0.0030
Epoch [2/5] | Batch [730/730] | Train Loss: 0.0060
--- Epoch 2 Complete | Train Loss: 0.0031 | Val Loss: 0.0025 | Time: 11.29 mins ---
Saved checkpoint to: /content/drive/MyDrive/hdemucs_sanidha_4stem_epoch5.pt
Epoch [3/5] | Batch [150/730] | Train Loss: 0.0029
Epoch [3/5] | Batch [300/730] | Train Loss: 0.0034
Epoch [3/5] | Batch [450/730] | Train Loss: 0.0049
Epoch [3/5] | Batch [600/730] | Train Loss: 0.0031
Epoch [3/5] | Batch [730/730] | Train Loss: 0.0016
--- Epoch 3 Complete | Train Loss: 0.0029 | Val Loss: 0.0024 | Time: 11.31 mins ---
Saved checkpoint to: /content/drive/MyDrive/hdemucs_sanidha_4stem_epoch5.pt
Epoch [4/5

In [7]:
# Cell 7: 4-Stem and Collapsed 2-Stem Evaluation on Concert 03
!pip install -q mir_eval

import mir_eval
import numpy as np

model.eval()
results_4stem = {"vocals": [], "violin": [], "mridangam": [], "ghatam": []}
results_2stem = {"vocals": [], "accompaniment": []}

eval_chunks = 25  # Evaluate across 25 active chunks
evaluated_count = 0

print(f"Evaluating 4-stem model on Concert 03 held-out chunks...")

with torch.no_grad():
    for idx in range(len(val_ds)):
        if evaluated_count >= eval_chunks:
            break

        mix, targets, mask = val_ds[idx]

        # Filter for active vocal sections
        if targets[0].pow(2).mean().sqrt() < 0.02:
            continue

        mix_cuda = mix.unsqueeze(0).to(device) # (1, 2, T)
        preds = model(mix_cuda).squeeze(0).cpu().numpy() # (4, 2, T)
        targets = targets.numpy() # (4, 2, T)

        # 1. 4-Stem SDR
        ref_4 = targets.reshape(4, -1)
        est_4 = preds.reshape(4, -1)
        sdr_4, _, _, _ = mir_eval.separation.bss_eval_sources(ref_4, est_4, compute_permutation=False)
        for i, k in enumerate(["vocals", "violin", "mridangam", "ghatam"]):
            results_4stem[k].append(sdr_4[i])

        # 2. Collapsed 2-Stem SDR (vocals vs sum of backing instruments)
        ref_voc = targets[0]
        ref_acc = targets[1:].sum(axis=0)
        est_voc = preds[0]
        est_acc = preds[1:].sum(axis=0)

        ref_2 = np.stack([ref_voc.reshape(-1), ref_acc.reshape(-1)])
        est_2 = np.stack([est_voc.reshape(-1), est_acc.reshape(-1)])
        sdr_2, _, _, _ = mir_eval.separation.bss_eval_sources(ref_2, est_2, compute_permutation=False)
        results_2stem["vocals"].append(sdr_2[0])
        results_2stem["accompaniment"].append(sdr_2[1])

        evaluated_count += 1

print("\n================== 4-STEM EVALUATION RESULTS (Concert 03) ==================")
for k in ["vocals", "violin", "mridangam", "ghatam"]:
    print(f"{k.capitalize():12s} SDR: {np.mean(results_4stem[k]):.2f} ± {np.std(results_4stem[k]):.2f} dB")
mean_4stem = np.mean([np.mean(results_4stem[k]) for k in results_4stem])
print(f"Mean 4-Stem SDR:    {mean_4stem:.2f} dB")

print("\n================== COLLAPSED 2-STEM BENCHMARK COMPARISON ==================")
print(f"Vocals SDR:         {np.mean(results_2stem['vocals']):.2f} ± {np.std(results_2stem['vocals']):.2f} dB")
print(f"Accompaniment SDR:  {np.mean(results_2stem['accompaniment']):.2f} ± {np.std(results_2stem['accompaniment']):.2f} dB")
mean_2stem = np.mean([np.mean(results_2stem['vocals']), np.mean(results_2stem['accompaniment'])])
print(f"Overall Mean SDR:   {mean_2stem:.2f} dB")
print(f"Comparison vs Sanidha Baseline (~7.87 dB): +{mean_2stem - 7.87:.2f} dB")

Evaluating 4-stem model on Concert 03 held-out chunks...


/tmp/ipykernel_4643/540467398.py:34: FutureWarning: mir_eval.separation.bss_eval_sources
	Deprecated as of mir_eval version 0.8.
	It will be removed in mir_eval version 0.9.
  sdr_4, _, _, _ = mir_eval.separation.bss_eval_sources(ref_4, est_4, compute_permutation=False)
/tmp/ipykernel_4643/540467398.py:46: FutureWarning: mir_eval.separation.bss_eval_sources
	Deprecated as of mir_eval version 0.8.
	It will be removed in mir_eval version 0.9.
  sdr_2, _, _, _ = mir_eval.separation.bss_eval_sources(ref_2, est_2, compute_permutation=False)



================== 4-STEM EVALUATION RESULTS (Concert 03) ==================
Vocals       SDR: 10.70 ± 6.02 dB
Violin       SDR: 6.31 ± 3.46 dB
Mridangam    SDR: 1.75 ± 11.16 dB
Ghatam       SDR: -6.08 ± 7.07 dB
Mean 4-Stem SDR:    3.17 dB

================== COLLAPSED 2-STEM BENCHMARK COMPARISON ==================
Vocals SDR:         10.70 ± 6.02 dB
Accompaniment SDR:  8.87 ± 2.90 dB
Overall Mean SDR:   9.78 dB
Comparison vs Sanidha Baseline (~7.87 dB): +1.91 dB


In [8]:
# Cell 8: Serialize 4-Stem Evaluation Metrics to JSON
import json

metrics_payload = {
    "evaluation_split": "Concert 03 (Held-Out Out-of-Distribution Validation)",
    "num_evaluated_chunks": evaluated_count,
    "individual_4stem": {
        "vocals_sdr_mean": float(np.mean(results_4stem["vocals"])),
        "vocals_sdr_std": float(np.std(results_4stem["vocals"])),
        "violin_sdr_mean": float(np.mean(results_4stem["violin"])),
        "violin_sdr_std": float(np.std(results_4stem["violin"])),
        "mridangam_sdr_mean": float(np.mean(results_4stem["mridangam"])),
        "mridangam_sdr_std": float(np.std(results_4stem["mridangam"])),
        "ghatam_sdr_mean": float(np.mean(results_4stem["ghatam"])),
        "ghatam_sdr_std": float(np.std(results_4stem["ghatam"])),
        "overall_mean_4stem_sdr": float(mean_4stem)
    },
    "collapsed_2stem": {
        "vocals_sdr_mean": float(np.mean(results_2stem["vocals"])),
        "vocals_sdr_std": float(np.std(results_2stem["vocals"])),
        "accompaniment_sdr_mean": float(np.mean(results_2stem["accompaniment"])),
        "accompaniment_sdr_std": float(np.std(results_2stem["accompaniment"])),
        "overall_mean_2stem_sdr": float(mean_2stem),
        "sanidha_paper_baseline": 7.87,
        "delta_vs_paper": float(mean_2stem - 7.87)
    }
}

out_path = "/content/drive/MyDrive/sanidha_4stem_concert03_metrics.json"
with open(out_path, "w") as f:
    json.dump(metrics_payload, f, indent=2)

print(f"Metrics JSON saved to: {out_path}")

Metrics JSON saved to: /content/drive/MyDrive/sanidha_4stem_concert03_metrics.json


In [9]:
# Cell 9: Export 4-Stem Separated Audio Files (.wav)
import soundfile as sf
import os

out_audio_dir = "/content/drive/MyDrive/sanidha_4stem_audio_samples"
os.makedirs(out_audio_dir, exist_ok=True)

# Select one representative active singing chunk from val_ds
mix, targets, mask = val_ds[10]  # Chunk 10 from Concert 03
mix_cuda = mix.unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    preds = model(mix_cuda).squeeze(0).cpu().numpy()  # (4, 2, T)
    targets = targets.numpy()                         # (4, 2, T)
    mix_np = mix.numpy()                              # (2, T)

sr = 44100
# Write Mix
sf.write(f"{out_audio_dir}/01_mix.wav", mix_np.T, sr, subtype="PCM_16")

# Write Ground-Truth Stems
sf.write(f"{out_audio_dir}/02_ref_vocals.wav", targets[0].T, sr, subtype="PCM_16")
sf.write(f"{out_audio_dir}/03_ref_violin.wav", targets[1].T, sr, subtype="PCM_16")
sf.write(f"{out_audio_dir}/04_ref_mridangam.wav", targets[2].T, sr, subtype="PCM_16")
sf.write(f"{out_audio_dir}/05_ref_ghatam.wav", targets[3].T, sr, subtype="PCM_16")

# Write Separated 4 Stems
sf.write(f"{out_audio_dir}/06_pred_vocals.wav", preds[0].T, sr, subtype="PCM_16")
sf.write(f"{out_audio_dir}/07_pred_violin.wav", preds[1].T, sr, subtype="PCM_16")
sf.write(f"{out_audio_dir}/08_pred_mridangam.wav", preds[2].T, sr, subtype="PCM_16")
sf.write(f"{out_audio_dir}/09_pred_ghatam.wav", preds[3].T, sr, subtype="PCM_16")

print(f"All 9 audio files successfully exported to {out_audio_dir}!")


All 9 audio files successfully exported to /content/drive/MyDrive/sanidha_4stem_audio_samples!


In [4]:
# Cell 10-A: Train TimbreCNN on IMIDataset (Takes ~3 mins on GPU)
!pip install -q kagglehub librosa soundfile

import os, kagglehub, librosa, random, torch
import numpy as np
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path

# 1. Download IMIDataset
print("Downloading IMIDataset via kagglehub...")
dataset_dir = kagglehub.dataset_download("patelp09/imidataset")
base_path = Path(dataset_dir)

CLASSES = ["bass", "drums", "flute", "guitar", "piano", "sarod", "sitar", "tabla", "veena", "violin"]
class_to_idx = {cls: idx for idx, cls in enumerate(CLASSES)}

TARGET_SR = 22050
TARGET_SAMPLES = int(TARGET_SR * 2.0)  # 2.0s duration = 44,100 samples
N_MFCC = 40
HOP_LENGTH = 512

def extract_feat(audio_path):
    y, sr = librosa.load(str(audio_path), sr=TARGET_SR, mono=True)
    if len(y) < TARGET_SAMPLES:
        y = np.pad(y, (0, TARGET_SAMPLES - len(y)), mode='constant')
    else:
        y = y[:TARGET_SAMPLES]
    return librosa.feature.mfcc(y=y, sr=TARGET_SR, n_mfcc=N_MFCC, hop_length=HOP_LENGTH).astype(np.float32)

# 2. Index audio tracks by class (handling singular 'drum' prefix)
all_wavs = list(base_path.glob("**/*.wav"))
by_class = {cls: [] for cls in CLASSES}

for w in all_wavs:
    prefix = w.name.split("_")[0].lower()
    if prefix == "drum":
        cls = "drums"
    elif prefix in class_to_idx:
        cls = prefix
    else:
        continue
    by_class[cls].append(w)

# 3. Extract balanced subset (100 samples per class = 1,000 total)
random.seed(42)
np.random.seed(42)

X_list, y_list = [], []
for cls in CLASSES:
    available = by_class[cls]
    chosen = random.sample(available, min(100, len(available)))
    print(f"Extracting features for {cls:10s} ({len(chosen)} tracks)...")
    for p in chosen:
        try:
            X_list.append(extract_feat(p))
            y_list.append(class_to_idx[cls])
        except Exception:
            continue

X = np.stack(X_list)  # (N, 40, 87)
y = np.array(y_list, dtype=np.int64)

# 4. Train / Validation Split (80 / 20)
indices = np.arange(len(y))
np.random.shuffle(indices)
split = int(0.8 * len(indices))
train_idx, val_idx = indices[:split], indices[split:]

# Add channel dimension (N, 1, 40, 87)
X_tensor = torch.from_numpy(X).unsqueeze(1)
y_tensor = torch.from_numpy(y)

train_loader = DataLoader(TensorDataset(X_tensor[train_idx], y_tensor[train_idx]), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_tensor[val_idx], y_tensor[val_idx]), batch_size=32, shuffle=False)

# 5. Architecture
class TimbreCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=10, dropout=0.3):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        if x.dim() == 3:
            x = x.unsqueeze(1)
        return self.fc(self.conv_blocks(x))

# 6. Training Loop (15 epochs)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classifier = TimbreCNN(num_classes=10).to(device)
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

best_acc = 0.0
save_path = "/content/drive/MyDrive/timbre_cnn_best.pt"

print(f"\nTraining TimbreCNN on {device}...")
for ep in range(1, 16):
    classifier.train()
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(classifier(bx), by)
        loss.backward()
        optimizer.step()

    classifier.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            preds = classifier(bx).argmax(dim=1)
            correct += (preds == by).sum().item()
            total += by.size(0)
    acc = correct / total
    if acc > best_acc:
        best_acc = acc
        torch.save(classifier.state_dict(), save_path)
    print(f"Epoch {ep:2d}/15 | Val Acc: {acc*100:.1f}% (Best: {best_acc*100:.1f}%)")

print(f"\nTraining complete! Checkpoint saved permanently to Google Drive: {save_path}")

Using Colab cache for faster access to the 'imidataset' dataset.
Extracting features for bass       (100 tracks)...
Extracting features for drums      (100 tracks)...
Extracting features for flute      (100 tracks)...
Extracting features for guitar     (100 tracks)...
Extracting features for piano      (100 tracks)...
Extracting features for sarod      (100 tracks)...
Extracting features for sitar      (100 tracks)...
Extracting features for tabla      (100 tracks)...
Extracting features for veena      (100 tracks)...
Extracting features for violin     (100 tracks)...

Training TimbreCNN on cpu...
Epoch  1/15 | Val Acc: 66.0% (Best: 66.0%)
Epoch  2/15 | Val Acc: 79.0% (Best: 79.0%)
Epoch  3/15 | Val Acc: 78.5% (Best: 79.0%)
Epoch  4/15 | Val Acc: 82.5% (Best: 82.5%)
Epoch  5/15 | Val Acc: 79.5% (Best: 82.5%)
Epoch  6/15 | Val Acc: 85.0% (Best: 85.0%)
Epoch  7/15 | Val Acc: 86.0% (Best: 86.0%)
Epoch  8/15 | Val Acc: 81.5% (Best: 86.0%)
Epoch  9/15 | Val Acc: 85.5% (Best: 86.0%)
Epoch 10

In [3]:
from google.colab import drive
import os

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')
else:
    print("Google Drive is mounted and accessible!")

Mounted at /content/drive


In [5]:
# Cell 10-B: 4-Stem Bleed Audit across Isolated Tracks
import torch
import torch.nn.functional as F
import numpy as np
import json
from pathlib import Path

# 1. Load trained classifier
classifier = TimbreCNN(num_classes=10).to(device)
classifier.load_state_dict(torch.load("/content/drive/MyDrive/timbre_cnn_best.pt", map_location=device))
classifier.eval()

# 2. Target audio tracks
stem_dir = Path("/content/drive/MyDrive/sanidha_4stem_audio_samples")
test_stems = [
    ("Polyphonic Mixture", stem_dir / "01_mix.wav"),
    ("Ref: Vocals",        stem_dir / "02_ref_vocals.wav"),
    ("Ref: Violin",        stem_dir / "03_ref_violin.wav"),
    ("Ref: Mridangam",     stem_dir / "04_ref_mridangam.wav"),
    ("Ref: Ghatam",        stem_dir / "05_ref_ghatam.wav"),
    ("Pred: Vocals",       stem_dir / "06_pred_vocals.wav"),
    ("Pred: Violin",       stem_dir / "07_pred_violin.wav"),
    ("Pred: Mridangam",    stem_dir / "08_pred_mridangam.wav"),
    ("Pred: Ghatam",       stem_dir / "09_pred_ghatam.wav"),
]

def classify_stem(path):
    if not path.exists():
        return None
    feat = extract_feat(str(path))
    tensor_in = torch.from_numpy(feat).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = classifier(tensor_in)
        probs = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    return probs

print(f"{'Audio Stem':22s} | {'Top Predicted':14s} | {'Confidence':10s} | {'Percussion Bleed':18s} | {'Melodic String Energy'}")
print("-" * 96)

audit_records = []
for name, path in test_stems:
    probs = classify_stem(path)
    if probs is None:
        print(f"{name:22s} | [FILE NOT FOUND: {path.name}]")
        continue
    top_idx = int(np.argmax(probs))
    top_cls = CLASSES[top_idx]
    top_p = probs[top_idx] * 100

    perc_bleed = (probs[class_to_idx["tabla"]] + probs[class_to_idx["drums"]]) * 100
    melodic_energy = (probs[class_to_idx["violin"]] + probs[class_to_idx["sitar"]] + probs[class_to_idx["veena"]]) * 100

    print(f"{name:22s} | {top_cls:14s} | {top_p:8.1f}%  | {perc_bleed:16.1f}% | {melodic_energy:20.1f}%")

    audit_records.append({
        "stem_name": name,
        "top_class": top_cls,
        "top_probability": float(probs[top_idx]),
        "percussion_energy_pct": float(perc_bleed),
        "melodic_energy_pct": float(melodic_energy),
        "full_distribution": {CLASSES[i]: float(probs[i]) for i in range(len(CLASSES))}
    })

# Save audit results to Drive
audit_out_path = "/content/drive/MyDrive/sanidha_4stem_timbre_audit.json"
with open(audit_out_path, "w") as f:
    json.dump({"milestone": "Milestone 4 - 4-Stem Bleed Audit", "evaluations": audit_records}, f, indent=2)

print(f"\nAudit results successfully saved to: {audit_out_path}")

Audio Stem             | Top Predicted  | Confidence | Percussion Bleed   | Melodic String Energy
------------------------------------------------------------------------------------------------
Polyphonic Mixture     | violin         |     83.7%  |              0.1% |                 84.0%
Ref: Vocals            | violin         |     59.9%  |              0.1% |                 60.0%
Ref: Violin            | violin         |     82.3%  |              0.0% |                 82.4%
Ref: Mridangam         | drums          |     43.4%  |             46.3% |                 27.4%
Ref: Ghatam            | drums          |     51.3%  |             52.0% |                 21.7%
Pred: Vocals           | violin         |     66.9%  |              0.3% |                 72.8%
Pred: Violin           | violin         |     88.2%  |              0.0% |                 88.7%
Pred: Mridangam        | sitar          |     28.2%  |             13.8% |                 54.1%
Pred: Ghatam           | viol

In [7]:
# Self-contained Audio Player (Reads directly from Google Drive)
import soundfile as sf
import IPython.display as ipd
from IPython.display import HTML, display
from pathlib import Path
from google.colab import drive
import os

# 1. Ensure Drive is mounted
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')

audio_dir = Path("/content/drive/MyDrive/sanidha_4stem_audio_samples")
assert audio_dir.exists(), f"Directory not found: {audio_dir}. Check Google Drive!"

tracks = [
    ("Original Polyphonic Mix (Input)", "01_mix.wav", "#2c3e50"),
    ("---", None, None),
    ("Fine-Tuned Pred: Vocals", "06_pred_vocals.wav", "#2980b9"),
    ("Ground-Truth: Vocals", "02_ref_vocals.wav", "#27ae60"),
    ("---", None, None),
    ("Fine-Tuned Pred: Violin (Melodic)", "07_pred_violin.wav", "#8e44ad"),
    ("Ground-Truth: Violin", "03_ref_violin.wav", "#27ae60"),
    ("---", None, None),
    ("Fine-Tuned Pred: Mridangam (Percussion)", "08_pred_mridangam.wav", "#d35400"),
    ("Ground-Truth: Mridangam", "04_ref_mridangam.wav", "#27ae60"),
    ("---", None, None),
    ("Fine-Tuned Pred: Ghatam (Clay Pot)", "09_pred_ghatam.wav", "#c0392b"),
    ("Ground-Truth: Ghatam", "05_ref_ghatam.wav", "#27ae60"),
]

display(HTML("<h2>Carnatic 4-Stem Audio Player</h2>"))

for title, fname, color in tracks:
    if title == "---":
        display(HTML("<hr style='margin: 12px 0; border: 0; border-top: 1px solid #ddd;'>"))
        continue

    fpath = audio_dir / fname
    if fpath.exists():
        data, sr = sf.read(fpath)
        display(HTML(f"""
        <div style="margin-top: 6px; margin-bottom: 2px;">
            <span style="background-color: {color}; color: white; padding: 3px 8px; border-radius: 4px; font-weight: bold; font-family: monospace; font-size: 13px;">
                {title}
            </span>
        </div>
        """))
        display(ipd.Audio(data.T, rate=sr))
    else:
        print(f"File not found: {fname}")